# Probe: LAZ write + FUSE copy + lidar_gbx round-trip (CPU)

Throwaway CPU probe to validate the light-tier point-cloud write path independent of GPU/dense: `write_xyzrgb_laz` (laspy + lazrs) -> local temp -> `shutil.copy` to a Volume (FUSE) -> read back via the `lidar_gbx` metadata reader. Returns the round-trip `point_count` via `dbutils.notebook.exit` so the result is retrievable from the job output (serverless stdout is not).

In [ ]:
%pip install "geobrix[light_env5] @ file:///Volumes/geospatial_docs/gdal_artifacts/noble/geobrix/geobrix-0.5.2-py3-none-any.whl"
%restart_python

In [ ]:
import json
import shutil
import tempfile
from pathlib import Path

import numpy as np

from databricks.labs.gbx.ds.register import register
from databricks.labs.gbx.pyrx.imagery import write_xyzrgb_laz

register(spark)  # registers lidar_gbx (+ others)

out = {}
try:
    import laspy
    out["laspy"] = laspy.__version__
    try:
        import lazrs  # noqa: F401
        out["lazrs"] = "present"
    except Exception as e:  # noqa: BLE001
        out["lazrs"] = f"MISSING: {type(e).__name__}"

    # 1) synthetic colored cloud
    n = 5000
    rng = np.random.default_rng(0)
    x = rng.uniform(-50, 50, n)
    y = rng.uniform(-50, 50, n)
    z = rng.uniform(-80, -30, n)
    r = rng.integers(0, 256, n).astype("uint8"); g = r.copy(); b = r.copy()

    vol_dir = "/Volumes/geospatial_docs/orthomosaic/data/gopro/outputs/_laz_probe"
    Path(vol_dir).mkdir(parents=True, exist_ok=True)
    out_laz = f"{vol_dir}/probe.laz"

    # 2) FUSE-safe write: local temp then copy (mirror dense_cloud_to_laz)
    _tmp = str(Path(tempfile.gettempdir()) / "probe.laz")
    written = write_xyzrgb_laz(_tmp, x, y, z, r, g, b, crs=None)
    out["local_written"] = written
    out["local_suffix"] = Path(written).suffix
    dst = str(Path(out_laz).with_suffix(Path(written).suffix))
    shutil.copy(written, dst)
    out["volume_dst"] = dst
    out["volume_bytes"] = Path(dst).stat().st_size

    # 3) read back via lidar_gbx metadata
    meta = spark.read.format("lidar_gbx").option("mode", "metadata").load(dst)
    row = meta.select("point_count", "x_min", "x_max", "z_min", "z_max", "crs").collect()[0]
    out["lidar_point_count"] = int(row["point_count"]) if row["point_count"] is not None else None
    out["z_range"] = [row["z_min"], row["z_max"]]
    out["input_n"] = n
    out["roundtrip_ok"] = out["lidar_point_count"] == n
except Exception as e:  # noqa: BLE001
    import traceback
    out["error"] = f"{type(e).__name__}: {e}"
    out["trace"] = traceback.format_exc()[-1500:]

print(json.dumps(out, indent=2))
dbutils.notebook.exit(json.dumps(out))